# Chicago Crime Analysis
## Requête 2 — Pattern Mining & Analyse Avancée
**Auteure : Ekta**

**Objectif :** Identifier des associations récurrentes entre les caractéristiques 
des crimes (type, moment de la journée, lieu, arrestation) grâce à l'algorithme Apriori.

---

In [6]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import plotly.graph_objects as go
import plotly.express as px

C:\Users\ektam\anaconda3\Lib\site-packages\plotly\express\imshow_utils.py:24: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  np.bool8: (False, True),


In [7]:
import plotly.io as pio
pio.renderers.default = "notebook"

In [8]:
import warnings
warnings.filterwarnings("ignore")

### Chargement du dataset

In [17]:
import sys
import pandas as pd

def load_data(limit=10000):

    url = (
        f"https://data.cityofchicago.org/resource/ijzp-q8t2.csv"
        f"?$limit={limit}"
        f"&$order=date%20DESC"
        f"&$where=latitude%20IS%20NOT%20NULL"
    )

    # Fix SSL uniquement sur Mac (Python 3.12 ne trouve pas les certificats système)
    if sys.platform == "darwin":
        import requests
        import io
        import certifi
        response = requests.get(url, verify=certifi.where())
        df = pd.read_csv(io.StringIO(response.text))
    else:
        df = pd.read_csv(url)

    # Renommage des colonnes (l'API retourne en minuscules)
    df = df.rename(columns={
        'latitude'            : 'Latitude',
        'longitude'           : 'Longitude',
        'primary_type'        : 'Primary Type',
        'arrest'              : 'Arrest',
        'date'                : 'Date',
        'location_description': 'Location Description',
        'domestic'            : 'Domestic',
        'beat'                : 'Beat',
        'ward'                : 'Ward',
        'fbi_code'            : 'FBI Code',
        'year'                : 'Year',
        'description'         : 'Description'
    })

    df["Date"] = pd.to_datetime(df["Date"], format="mixed")

    print(f"Dataset chargé : {df.shape[0]} lignes × {df.shape[1]} colonnes")
    return df

df = load_data()
print("Dataset chargé avec succès")
df.head()

Dataset chargé : 10000 lignes × 22 colonnes
Dataset chargé avec succès


,id,case_number,Date,block,iucr,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,community_area,FBI Code,x_coordinate,y_coordinate,Year,updated_on,Latitude,Longitude,location
0,14223030,JK286094,2026-06-08,036XX W DOUGLAS BLVD,0497,BATTERY,AGGRAVATED DOMESTIC BATTERY - OTHER DANGEROUS ...,APARTMENT,False,True,...,24,29,04B,1152476,1893218,2026,2026-06-15T15:46:40.000,41.862851,-87.715755,"\n, \n(41.862850968, -87.715755)"
1,14223973,JK287246,2026-06-08,072XX S RICHMOND ST,0486,BATTERY,DOMESTIC BATTERY SIMPLE,RESIDENCE,False,True,...,18,66,08B,1157959,1856471,2026,2026-06-15T15:46:40.000,41.761902,-87.696627,"\n, \n(41.76190249, -87.696626851)"
2,14223471,JK286728,2026-06-08,076XX S MORGAN ST,0910,MOTOR VEHICLE THEFT,AUTOMOBILE,STREET,False,False,...,17,71,07,1170965,1854265,2026,2026-06-15T15:46:40.000,41.755575,-87.649023,"\n, \n(41.755574885, -87.649022623)"
3,14224250,JK286700,2026-06-08,061XX N KIRKWOOD AVE,0930,MOTOR VEHICLE THEFT,THEFT / RECOVERY - AUTOMOBILE,STREET,False,False,...,39,12,07,1145138,1940565,2026,2026-06-15T15:46:40.000,41.992917,-87.741493,"\n, \n(41.992917396, -87.741492591)"
4,14225502,JK287707,2026-06-08,040XX W DICKENS AVE,2825,OTHER OFFENSE,HARASSMENT BY TELEPHONE,RESIDENCE,False,False,...,35,20,26,1148906,1913645,2026,2026-06-15T15:46:40.000,41.918975,-87.728331,"\n, \n(41.918974575, -87.728331482)"


## Étape 1 : Discrétisation des variables

Avant d'appliquer l'algorithme Apriori, on doit transformer les variables continues 
ou textuelles en **catégories discrètes** :

| Variable | Transformation |
|---|---|
| `Date` (heure) | matin (6h-12h) / après-midi (12h-18h) / soir (18h-23h) / nuit (0h-6h) |
| `Primary Type` | gardé tel quel (type de crime) |
| `Location Description` | simplifié en grandes catégories (rue, intérieur, transport...) |
| `Arrest` | Arrestation_OUI / Arrestation_NON |
| `Domestic` | Domestique_OUI / Domestique_NON |

Cette étape est indispensable car Apriori travaille uniquement sur des données **booléennes**.

In [18]:
#fonction de discrétisation des variables
# Input : DataFrame brut
# Output : DataFrame avec colonnes discrétisées

def discretiser(df):
    df = df.copy()
    
    # 1. Discrétisation de l'heure en 4 tranches
    heure = df["Date"].dt.hour
    def tranche_horaire(h):
        if 6 <= h < 12:
            return "Matin"
        elif 12 <= h < 18:
            return "Après-midi"
        elif 18 <= h < 23:
            return "Soir"
        else:
            return "Nuit"
    df["Heure"] = heure.apply(tranche_horaire)
    
    # 2. Simplification du lieu (garder les 5 lieux les plus fréquents, regrouper le reste)
    top_lieux = df["Location Description"].value_counts().nlargest(5).index
    df["Lieu"] = df["Location Description"].apply(
        lambda x: x if x in top_lieux else "AUTRE"
    )
    
    # 3. Arrestation en texte lisible
    df["Arrestation"] = df["Arrest"].apply(lambda x: "Arrestation_OUI" if x else "Arrestation_NON")
    
    # 4. Crime domestique
    df["Domestique"] = df["Domestic"].apply(lambda x: "Domestique_OUI" if x else "Domestique_NON")

    df_disc = df[["Primary Type", "Heure", "Lieu", "Arrestation", "Domestique"]]
    
    return df_disc

df_disc = discretiser(df)
print("Discrétisation terminée")
print(df_disc["Heure"].value_counts())
print(df_disc["Lieu"].value_counts())
df_disc.head()

Discrétisation terminée
Heure
Après-midi    2841
Nuit          2646
Soir          2538
Matin         1975
Name: count, dtype: int64
Lieu
STREET                3152
AUTRE                 3076
APARTMENT             1827
RESIDENCE             1064
SIDEWALK               529
SMALL RETAIL STORE     352
Name: count, dtype: int64


,Primary Type,Heure,Lieu,Arrestation,Domestique
0,BATTERY,Nuit,APARTMENT,Arrestation_NON,Domestique_OUI
1,BATTERY,Nuit,RESIDENCE,Arrestation_NON,Domestique_OUI
2,MOTOR VEHICLE THEFT,Nuit,STREET,Arrestation_NON,Domestique_NON
3,MOTOR VEHICLE THEFT,Nuit,STREET,Arrestation_NON,Domestique_NON
4,OTHER OFFENSE,Nuit,RESIDENCE,Arrestation_NON,Domestique_NON


### Résultats de la discrétisation

**Répartition temporelle :**
- L'après-midi (12h-18h) est la tranche horaire la plus criminogène avec 2841 incidents (28%)
- La nuit (0h-6h) arrive en 2ème position avec 2646 incidents  car ces crimes sont souvent plus difficiles à résoudre
- Le matin reste la période la plus calme avec 1975 incidents

**Lieux les plus fréquents :**
- La **rue (STREET)** est le lieu le plus dangereux avec 3152 incidents
- Les **appartements (APARTMENT)** arrivent en 3ème position avec 1827 incidents,
  ce qui suggère une forte présence de crimes domestiques
- Les petits commerces (SMALL RETAIL STORE) ferment la liste avec 352 incidents

In [19]:
# Fonction d'encodage en format binaire (one-hot)
# Input : DataFrame discrétisé
# Output : DataFrame binaire prêt pour Apriori

def encoder_transactions(df_disc):
    #transforme chaque ligne en liste d'items
    transactions = df_disc.apply(lambda row: list(row.values), axis=1).tolist()
    
    te = TransactionEncoder()
    te_array = te.fit(transactions).transform(transactions)
    df_encoded = pd.DataFrame(te_array, columns=te.columns_)
    
    return df_encoded

df_encoded = encoder_transactions(df_disc)
print(f"Matrice encodée : {df_encoded.shape[0]} lignes × {df_encoded.shape[1]} colonnes")
df_encoded.head()

Matrice encodée : 10000 lignes × 42 colonnes


,APARTMENT,ARSON,ASSAULT,AUTRE,Après-midi,Arrestation_NON,Arrestation_OUI,BATTERY,BURGLARY,CONCEALED CARRY LICENSE VIOLATION,...,RESIDENCE,ROBBERY,SEX OFFENSE,SIDEWALK,SMALL RETAIL STORE,STALKING,STREET,Soir,THEFT,WEAPONS VIOLATION
0,True,False,False,False,False,True,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,True,False,True,False,False,...,True,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
3,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
4,False,False,False,False,False,True,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False


## Étape 2 : Algorithme Apriori & Règles d'association



L'algorithme Apriori cherche des combinaisons d'items qui apparaissent souvent 
ensemble dans le dataset. On l'applique ici pour extraire des itemsets fréquents, 
puis générer des règles d'association mesurées par confidence et lift.

Nous choisissons **σ = 0.05** comme support minimal, ce qui signifie qu'une association 
doit apparaître dans au moins 5% des transactions, soit environ 500 incidents sur 10 000.
Ce seuil est un bon compromis : suffisamment bas pour capturer des patterns variés 
(119 itemsets, 324 règles), mais assez élevé pour éliminer les coïncidences rares 
non significatives.

In [24]:
# Fonction d'extraction des itemsets fréquents et règles d'association
# Input : DataFrame encodé, support minimal
# Output : DataFrame des règles d'association

def appliquer_apriori(df_encoded, min_support=0.05):
    # Extraction des itemsets fréquents
    itemsets = apriori(df_encoded, min_support=min_support, use_colnames=True)
    itemsets = itemsets.sort_values("support", ascending=False)
    
    # Génération des règles d'association
    regles = association_rules(itemsets, metric="lift", min_threshold=1.0)
    regles = regles.sort_values("lift", ascending=False)
    
    print(f"Itemsets fréquents trouvés : {len(itemsets)}")
    print(f"Règles d'association trouvées : {len(regles)}")
    
    return itemsets, regles

itemsets, regles = appliquer_apriori(df_encoded, min_support=0.05)
regles[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)

Itemsets fréquents trouvés : 119
Règles d'association trouvées : 324


,antecedents,consequents,support,confidence,lift
316,"frozenset({APARTMENT, BATTERY})",frozenset({Domestique_OUI}),0.0512,0.842105,4.311855
317,frozenset({Domestique_OUI}),"frozenset({APARTMENT, BATTERY})",0.0512,0.262161,4.311855
274,frozenset({MOTOR VEHICLE THEFT}),"frozenset({Domestique_NON, Arrestation_NON, ST...",0.0589,0.721814,2.909366
263,"frozenset({Domestique_NON, Arrestation_NON, ST...",frozenset({MOTOR VEHICLE THEFT}),0.0589,0.237404,2.909366
319,frozenset({BATTERY}),"frozenset({Domestique_OUI, APARTMENT})",0.0512,0.251597,2.764803
314,"frozenset({Domestique_OUI, APARTMENT})",frozenset({BATTERY}),0.0512,0.562637,2.764803
269,"frozenset({Domestique_NON, MOTOR VEHICLE THEFT})","frozenset({Arrestation_NON, STREET})",0.0589,0.734414,2.718038
268,"frozenset({Arrestation_NON, STREET})","frozenset({Domestique_NON, MOTOR VEHICLE THEFT})",0.0589,0.217987,2.718038
53,frozenset({BATTERY}),frozenset({Domestique_OUI}),0.1075,0.528256,2.704841
52,frozenset({Domestique_OUI}),frozenset({BATTERY}),0.1075,0.550435,2.704841


### Résultats de l'algorithme Apriori (σ=0.05)

L'algorithme a extrait **119 itemsets fréquents** et généré **324 règles d'association**.

### Top règles par lift — Interprétation

**Règle 1 — La plus forte (lift = 4.31) :**
> "Si BATTERY dans un APARTMENT → crime domestique"
- Confidence = 0.84 : dans 84% des cas, une agression en appartement est un crime domestique
- Lift = 4.31 : cette association est 4x plus fréquente que par hasard → très significatif
- Interprétation : les violences physiques en appartement sont quasi-systématiquement 
  des affaires familiales ou conjugales

**Règle 2 (lift = 2.91) :**
> "Si MOTOR VEHICLE THEFT → crime non domestique, sans arrestation, dans la rue"
- Confidence = 0.72 : dans 72% des cas, un vol de voiture a lieu dans la rue sans arrestation
- Lift = 2.91 : association 3x plus forte que le hasard
- Interprétation : les vols de véhicules sont des crimes opportunistes commis dans 
  l'espace public et rarement résolus par une arrestation immédiate

**Règle 3 (lift = 2.70) :**
> "Si BATTERY → crime domestique"
- Support = 0.107 : concerne plus de 10% de tous les incidents
- Interprétation : les agressions physiques so!nt fortement liées au cadre familial 
  à Chicago, ce qui pointe vers un problème structurel de violences conjugales/familiales

### Conclusion générale
Le pattern mining révèle deux profils criminels distincts :
- Les crimes **en intérieur (appartement)** → violence domestique, difficile à détecter
- Les crimes **en extérieur (rue)** → vols de véhicules, rarement élucidés sur le moment

## Impact du support minimal sur le nombre de patterns

On teste différentes valeurs de support minimal σ et on trace le nombre d'itemsets fréquents trouvés pour chaque valeur.
Ceci nous permet de choisir un bon compromis entre quantité et pertinence des règles.

In [27]:
# Fonction pour analyser l'impact du support minimal sur le nombre de patterns
# Input : DataFrame encodé
# Output : graphique Plotly

def graphique_support(df_encoded):
    supports = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4]
    nb_itemsets = []
    
    for s in supports:
        items = apriori(df_encoded, min_support=s, use_colnames=True)
        nb_itemsets.append(len(items))
    
    fig = px.line(
        x=supports,
        y=nb_itemsets,
        markers=True,
        title="Impact du support minimal σ sur le nombre d'itemsets fréquents",
        labels={"x": "Support minimal σ", "y": "Nombre d'itemsets fréquents"}
    )
    fig.add_vline(
        x=0.05, 
        line_dash="dash", 
        line_color="red",
        annotation_text="σ choisi = 0.05"
    )
    fig.show()
    return fig

fig_support = graphique_support(df_encoded)

Le graphique montre une chute brutale entre σ=0.05 (119 itemsets) et σ=0.1 (44 itemsets).
Au-delà de σ=0.1, la courbe s'aplatit progressivement jusqu'à seulement 3 itemsets à σ=0.4.

Nous avons choisi σ=0.05 car c'est le meilleur compromis :
- Assez bas pour capturer des associations intéressantes (119 itemsets, 324 règles)
- Assez élevé pour éliminer les coïncidences rares non significatives
- Correspond à des items présents dans au moins ~500 transactions sur 10 000

## Étape 3 — Visualisation Sankey

Le **diagramme de Sankey** permet de visualiser les règles d'association :
Dans ce graph les antécédents sont à gauche, les conséquents à droite.
L'épaisseur des liens représente la confidence, la couleur représente le lift.
On garde les top_n règles triées par lift pour ne garder que les plus intéressantes.

In [29]:
# Fonction de visualisation Sankey des règles d'association
# Input : DataFrame des règles
# Output : graphique Sankey Plotly

def sankey_regles(regles, top_n=15):
    """
    Visualise les top_n règles d'association sous forme de diagramme Sankey.
    Les antécédents sont à gauche, les conséquents à droite.
    On filtre les règles réciproques pour éviter les boucles.
    """
    top_regles = regles.nlargest(top_n, "lift").copy()
    
    top_regles["ant_str"] = top_regles["antecedents"].apply(lambda x: ", ".join(sorted(list(x))))
    top_regles["cons_str"] = top_regles["consequents"].apply(lambda x: ", ".join(sorted(list(x))))
    
    # Filtrer les boucles (antécédent == conséquent)
    top_regles = top_regles[top_regles["ant_str"] != top_regles["cons_str"]]
    
    # Filtrer les règles réciproques (garder seulement la meilleure direction)
    seen = set()
    rows = []
    for _, row in top_regles.iterrows():
        pair = frozenset([row["ant_str"], row["cons_str"]])
        if pair not in seen:
            seen.add(pair)
            rows.append(row)
    top_regles = pd.DataFrame(rows)
    
    all_labels = list(set(top_regles["ant_str"].tolist() + top_regles["cons_str"].tolist()))
    label_idx = {label: i for i, label in enumerate(all_labels)}
    
    sources = [label_idx[a] for a in top_regles["ant_str"]]
    targets = [label_idx[c] for c in top_regles["cons_str"]]
    values = top_regles["confidence"].tolist()
    lifts = top_regles["lift"].tolist()
    
    colors = [f"rgba(255, {max(0, int(255 - l*30))}, 0, 0.5)" for l in lifts]
    
    fig = go.Figure(go.Sankey(
        node=dict(
            pad=30,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=all_labels,
            color="lightblue"
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=colors
        )
    ))
    
    fig.update_layout(
        title_text="Règles d'association — Diagramme de Sankey (épaisseur = confidence, couleur = lift)",
        font_size=11,
        height=600,
        width=1000
    )
    fig.show()
    return fig

fig_sankey = sankey_regles(regles, top_n=15)

### Lecture du diagramme de Sankey

Chaque flux représente une règle d'association :
- **Gauche** = antécédent (condition SI)
- **Droite** = conséquent (conclusion ALORS)
- **Épaisseur du lien** = confidence (plus c'est épais, plus la règle est fiable)
- **Couleur** = lift (orange foncé = association très forte, jaune = association modérée)

**Ce qu'on observe :**
- **MOTOR VEHICLE THEFT** → mène systématiquement vers une absence d'arrestation 
  dans la rue (lien très épais = confidence élevée)
- **BATTERY + Domestique_OUI** → fortement associé aux appartements, 
  confirmant le profil de violence conjugale/familiale
- **APARTMENT, BATTERY** → transite par **Domestique_OUI** avant de mener 
  vers d'autres caractéristiques, montrant que ce nœud est central dans 
  les crimes domestiques

In [31]:
# Bloc principal — appelle toutes les fonctions dans l'ordre
if __name__ == "__main__":
    # Étape 1 : Chargement depuis l'API Chicago
    df = load_data()
    
    # Étape 2 : Discrétisation
    df_disc = discretiser(df)
    
    # Étape 3 : Encodage
    df_encoded = encoder_transactions(df_disc)
    
    # Étape 4 : Apriori
    itemsets, regles = appliquer_apriori(df_encoded, min_support=0.05)
    
    # Étape 5 : Visualisations
    fig_support = graphique_support(df_encoded)
    fig_sankey = sankey_regles(regles, top_n=15)
    
    print("✅ Pattern Mining terminé !")

Dataset chargé : 10000 lignes × 22 colonnes
Itemsets fréquents trouvés : 119
Règles d'association trouvées : 324


✅ Pattern Mining terminé !


## Conclusion Pattern Mining

Cette analyse par règles d'association a permis d'identifier des patterns récurrents 
dans la criminalité de Chicago sur 10 000 incidents récents.

Les principaux enseignements sont :
- Les **agressions (BATTERY) en appartement** sont quasi-systématiquement des crimes 
  domestiques (lift = 4.31): ce qui pointe vers un problème structurel de violences familiales
- Les **vols de véhicules (MOTOR VEHICLE THEFT)** ont lieu dans la rue sans arrestation 
  dans 72% des cas , des crimes opportunistes difficiles à résoudre immédiatement
- Les crimes commis **l'après-midi** sont les plus fréquents avec 2841 incidents

Ces patterns constituent une base solide pour anticiper les zones et situations à risque, 
et peuvent aider à mieux orienter les ressources policières sur le terrain.

**Prompt LLM utilisé :** Ce notebook a été développé avec l'assistance de Claude (Anthropic) 
pour la structure du code, les visualisations et les interprétations des métriques Apriori.